# 1. Imports

In [87]:
import numpy as np
import pandas as pd

from sklearn.model_selection import LeaveOneOut
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 2. Load data 

In [88]:
TICKETS_PATH = "../../data/raw/gold_match_tickets.csv"
CONTEXT_PATH = "../../data/raw/gold_match_context.csv"
TRENDS_PATH = "../../data/raw/gold_google_trends_daily.csv"
ARTICLES_PATH = "../../data/raw/gold_belga_press_articles.csv"
MATCH_BASE_PATH = "../../data/raw/gold_match.csv"  

df_tickets = pd.read_csv(TICKETS_PATH)
df_context = pd.read_csv(CONTEXT_PATH)
df_trends = pd.read_csv(TRENDS_PATH)
df_articles = pd.read_csv(ARTICLES_PATH, on_bad_lines='skip')
df_match = pd.read_csv(MATCH_BASE_PATH)

# 3. Feature Engineering - Tickets

In [90]:
tickets_features = df_tickets[
    ['match_id','tickets_sold_b2c','tickets_sold_b2b','seasonpass_holders','tickets_sold_total']
].copy()

tickets_features['pct_b2c_of_total'] = (
    tickets_features['tickets_sold_b2c'] / tickets_features['tickets_sold_total']
).fillna(0)

# 4. Feature Engineering — Match Context

In [ ]:
# 4. Feature Engineering — Match Context
context_features = df_context[
    ['match_id','pct_free_tickets','promo_tickets_total','has_promotion']
].copy()

df_tickets = df_tickets.merge(
    df_context[['match_id', 'match_date']],
    on='match_id',
    how='left'
)
df_tickets['match_date'] = pd.to_datetime(df_tickets['match_date'])

# 5. Feature Engineering — Google Trends

In [92]:
df_trends['date'] = pd.to_datetime(df_trends['date'])

# Merge match_date (nu in df_tickets) naar trends
df_trends = df_trends.merge(df_tickets[['match_id','match_date']], on='match_id', how='left')

# Compute days_to_match
df_trends['days_to_match'] = (df_trends['match_date'] - df_trends['date']).dt.days

# Aggregate 7-day average per match
trends_filtered = df_trends[
    (df_trends['days_to_match'] >= 1) & (df_trends['days_to_match'] <= 7)
]

trends_agg = (
    trends_filtered
    .groupby('match_id')['ohl_interest']
    .mean()
    .reset_index()
    .rename(columns={'ohl_interest': 'trends_7d_avg'})
)

# 6. Feature Engineering — Press Articles

In [93]:
articles_filtered = df_articles[
    (df_articles['days_to_match'] >= 1) 
    & (df_articles['days_to_match'] <= 14) 
    & (df_articles['match_id'].notna())
]

articles_agg = (
    articles_filtered
    .groupby('match_id').size()
    .reset_index(name='article_count_14d')
)

# 7. Merge All Feature Blocks

In [ ]:
df_model = tickets_features.merge(context_features, on='match_id', how='left')
df_model = df_model.merge(trends_agg, on='match_id', how='left')
df_model = df_model.merge(articles_agg, on='match_id', how='left')


df_model = df_model.merge(
    df_match[['match_id', 'tickets_scanned']],
    on='match_id',
    how='left'
)

# Fill missing values
df_model = df_model.fillna(0)

# 8. Define Features and Target

In [95]:
features_external = [
    'tickets_sold_b2c',
    'tickets_sold_b2b',
    'seasonpass_holders',
    'pct_b2c_of_total',
    'pct_free_tickets',
    'promo_tickets_total',
    'trends_7d_avg',
    'article_count_14d',
]

target = 'tickets_scanned'

X = df_model[features_external]
y = df_model[target]

# 9. Validation Strategy — LOOCV

In [96]:
loo = LeaveOneOut()

models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "RandomForest": RandomForestRegressor(max_depth=5, n_estimators=100, random_state=42)
}

results = []
rf_importances = []

for model_name, model in models.items():
    y_true, y_pred = [], []

    for train_idx, test_idx in loo.split(X):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # Standard scaling
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        # Fit
        model.fit(X_train_scaled, y_train)

        # Predict
        pred = model.predict(X_test_scaled)[0]
        y_true.append(y_test.values[0])
        y_pred.append(pred)

        # RF importance per fold
        if model_name == "RandomForest":
            rf_importances.append(model.feature_importances_)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((np.array(y_true) - np.array(y_pred)) / np.array(y_true))) * 100

    results.append({
        'Model': model_name,
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2,
        'MAPE': mape
    })

# 10. Results Table

In [97]:
results_df = pd.DataFrame(results).sort_values(by='R2', ascending=False)
print(results_df)

              Model          MAE          RMSE        R2       MAPE
2      RandomForest  1320.145236  2.314449e+06  0.410854  20.287184
1             Ridge  1495.887481  3.112586e+06  0.207686  22.713590
0  LinearRegression  1485.445848  3.128651e+06  0.203597  22.505183


# 11. Random Forest Feature Importance

In [98]:
# Mean importance across LOOCV folds
rf_mean_importance = np.mean(rf_importances, axis=0)
feature_importance_df = pd.DataFrame({
    'feature': features_external,
    'importance': rf_mean_importance
}).sort_values(by='importance', ascending=False)

print(feature_importance_df)

               feature  importance
0     tickets_sold_b2c    0.286284
1     tickets_sold_b2b    0.215240
4     pct_free_tickets    0.189548
3     pct_b2c_of_total    0.094987
2   seasonpass_holders    0.085977
5  promo_tickets_total    0.065545
7    article_count_14d    0.062420
6        trends_7d_avg    0.000000


# 12. Key Interpretation

In [102]:
# Best model R²
best_model_row = results_df.iloc[0]
print(f"Best model: {best_model_row['Model']}")
print(f"Best R²: {best_model_row['R2']:.3f}")

# Top RF feature
top_feature = feature_importance_df.iloc[0]
print(f"Top RF feature: {top_feature['feature']}")
print(f"Top RF feature importance: {top_feature['importance']:.3f}")

# pct_free_tickets importance
pct_row = feature_importance_df[feature_importance_df['feature'] == 'pct_free_tickets'].iloc[0]
pct_rank = feature_importance_df.index.get_loc(pct_row.name) + 1
print(f"pct_free_tickets importance: {pct_row['importance']:.3f}")
print(f"pct_free_tickets rank: {pct_rank}")

Best model: RandomForest
Best R²: 0.411
Top RF feature: tickets_sold_b2c
Top RF feature importance: 0.286
pct_free_tickets importance: 0.190
pct_free_tickets rank: 3
